In [ ]:
# Setup. Everything below runs the CLI itself, so the notebook and the PBS job
# execute identical code.
import logging
import os
import sys

for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from IPython.display import Image, Markdown, display  # noqa: E402

from scripts.run_isc import _build_arg_parser, run  # noqa: E402
from src.definitions.constants import ProjectPaths  # noqa: E402

logging.basicConfig(
    level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True
)
print("Setup complete.")

# Stage 02 — Inter-subject correlation

Leave-one-out, pairwise and sliding-window ISC. LOO-ISC is the default reading: it compares a subject against the group rather than against one other person.

One run covers both passes -- the unfiltered signal and each of the five bands -- because synchrony is not band-flat and the script has no flag to ask for only one.

**The method is documented in
[`docs/analyses/02-isc-broadband-analysis/`](../../docs/analyses/02-isc-broadband-analysis/README.md)** — this
notebook does not restate it.

**What this notebook is for.** Choosing parameters and checking the workflow cheaply
before committing a node to it. It calls `run()` from
[`scripts/run_isc.py`](../../scripts/run_isc.py) — the same function the PBS job
calls — at reduced scale, then shows what it wrote.

## Configuration

In [ ]:
# Small on purpose -- the full-size run is the job's.
CONDITION = "Placebo"
MUSIC_TYPE = "CLASSIC"   # or "PSYTRANCE"

# Subset every axis. A sliding window must fit inside what is kept: at 250 Hz the 15 s
# default needs 3,750 samples, so this is well above that.
N_SUBJECTS = 6
N_CHANNELS = 32
N_TIMES = 15000

OUT_DIR = ProjectPaths.NOTEBOOKS_DIR / "02-isc-broadband-analysis" / "out" / "isc"
PLOTS_DIR = OUT_DIR / "plots"

## Run

The same function the PBS job calls. Keep the scale small here: the full-size pass is what
[`jobs/metacentrum/02-isc-broadband-analysis/`](../../jobs/metacentrum/02-isc-broadband-analysis) is for.

In [ ]:
args = _build_arg_parser().parse_args(
    [
        "--condition", CONDITION,
        "--music_type", MUSIC_TYPE,
        "--n_subjects", str(N_SUBJECTS),
        "--n_channels", str(N_CHANNELS),
        "--n_times", str(N_TIMES),
        "--save_dir", str(PLOTS_DIR),
    ]
)
run(args)

## Figures

Written by the shared plotters in `src/visualization/`, which the CLI uses too --
so what is shown here is what the job produces.

In [ ]:
for png in sorted(PLOTS_DIR.rglob("*.png")):
    display(Markdown(f"**{png.relative_to(PLOTS_DIR)}**"))
    display(Image(filename=str(png)))